# VN Live-Commerce Host — Colab Bootstrap

Clone repo → install deps → download weights → run production `core` `/api/v1` backend → expose via ngrok.

**Models (default):**
- LLM: `gemma-3-4b-it` GGUF Q4_K_M (Gemma terms, gated — accept on HF first)
- TTS: `VieNeu-TTS-v2` (Apache-2.0, Vietnamese-native)

**Architecture:** backend = CONTROL plane (JSON + WS). Avatar VIDEO = LiveAvatar-cloud → LiveKit → browser (media plane, frames never transit backend).

**Runtime:** pick a **GPU** runtime (T4 free tier works for 4B Q4 + VieNeu).

## 1. Clone the repo

In [ ]:
REPO_URL = "https://github.com/<you>/<repo>.git"  # <-- EDIT
REPO_DIR = "/content/repo"
IMPL_DIR = REPO_DIR + "/projects/ai-livestream-commerce-vn/implementations"

import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    print("repo already cloned")
%cd $IMPL_DIR
!ls

## 2. Install dependencies

Core backend + llama.cpp (GGUF LLM) + VieNeu TTS + ngrok.

In [ ]:
!pip install -q -r liveavatar_api/requirements.txt
!pip install -q huggingface_hub llama-cpp-python pyngrok
# VieNeu-TTS — check the model card for the exact pip name.
# Try the official package; if it differs, adjust here.
!pip install -q vieneu-tts || echo 'VieNeu pip name may differ — check model card on HF'
# CMake for llama-cpp-python GPU build (T4):
!pip install -q cmake

## 3. Download model weights

- **LLM:** `gemma-3-4b-it` GGUF Q4_K_M (~2.5GB, fits T4 VRAM)
- **TTS:** `VieNeu-TTS-v2` (Apache-2.0, Vietnamese-native)

> ⚠️ gemma-3-4b-it is **gated** on HF. Go to https://huggingface.co/google/gemma-3-4b-it and accept the license BEFORE running this cell. Login with `hf auth login` or set `HF_TOKEN` in Colab Secrets.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download, login
import os

# HF login (for gated gemma model)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets for gated models:", e)

# --- LLM: gemma-3-4b-it GGUF Q4_K_M ---
LLM_REPO = "google/gemma-3-4b-it-GGUF"  # official GGUF repo (check exact name on HF)
LLM_FILE = os.environ.get("LLM_GGUF_FILE", "gemma-3-4b-it-Q4_K_M.gguf")

print(f"Downloading LLM: {LLM_REPO}/{LLM_FILE}...")
llm_path = hf_hub_download(repo_id=LLM_REPO, filename=LLM_FILE, local_dir="/content/weights/llm")
print("LLM GGUF:", llm_path)

# --- TTS: VieNeu-TTS-v2 ---
TTS_REPO = os.environ.get("TTS_REPO", "pnnbao-ump/VieNeu-TTS-v2")
print(f"Downloading TTS: {TTS_REPO}...")
tts_dir = snapshot_download(repo_id=TTS_REPO, local_dir="/content/weights/tts")
print("TTS dir:", tts_dir)

## 4. Move weights into repo layout

In [ ]:
import shutil, os
os.makedirs("weights/llm", exist_ok=True)
os.makedirs("weights/tts", exist_ok=True)

if llm_path:
    dst = os.path.join("weights/llm", os.path.basename(llm_path))
    if not os.path.exists(dst):
        shutil.copy(llm_path, dst)
    print("LLM ->", dst)

if os.path.isdir(tts_dir) and not os.listdir("weights/tts"):
    shutil.copytree(tts_dir, "weights/tts", dirs_exist_ok=True)
print("weights/:", os.listdir("weights"))

## 5. Set environment (secrets + engine selection)

Engine selection via env — the server auto-builds LLM + TTS from these:
- `LLM_ENGINE=llamacpp` (GGUF on T4)
- `LLM_GGUF_DIR=weights/llm` (auto-finds .gguf)
- `TTS_ENGINE=vieneu` (VN-native NeuTTS)
- `DIRECTOR_ENABLED=1` (enable the chat-cluster Director)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["LIVEAVATAR_API_KEY"] = userdata.get("LIVEAVATAR_API_KEY")
    os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")
except Exception as e:
    print("Set secrets manually:", e)

os.environ["RENDER_BACKEND"] = "cloud"
os.environ["SESSION_STORE"] = "memory"
os.environ["PORT"] = "8800"

# LLM: llama.cpp GGUF (gemma-3-4b-it Q4_K_M on T4)
os.environ["LLM_ENGINE"] = "llamacpp"
os.environ["LLM_GGUF_DIR"] = "weights/llm"
os.environ["LLM_N_CTX"] = "4096"
os.environ["LLM_N_GPU_LAYERS"] = "-1"  # all layers on GPU
os.environ["LLM_MAX_TOKENS"] = "128"

# TTS: VieNeu-TTS (VN-native)
os.environ["TTS_ENGINE"] = "vieneu"
os.environ["TTS_WEIGHTS"] = "weights/tts"
os.environ["TTS_DEVICE"] = "cuda"

# Director (chat cluster + scoring FSM)
os.environ["DIRECTOR_ENABLED"] = "1"

print("env set. key loaded:", bool(os.environ.get("LIVEAVATAR_API_KEY")))
print("LLM engine:", os.environ.get("LLM_ENGINE"))
print("TTS engine:", os.environ.get("TTS_ENGINE"))

## 6. Sandbox smoke test (free, no credits)

Verify the API + LiveAvatar key work BEFORE loading models.

In [ ]:
# This uses the echo/tone stubs (models not loaded yet) — just tests the API surface.
!LLM_ENGINE=none TTS_ENGINE=tone python -m core.tests.v1_smoke_test

## 7. Launch backend + ngrok

Loads gemma-3-4b-it GGUF + VieNeu-TTS, injects into the cloud RenderBackend, serves `/api/v1`, opens ngrok.

**This cell blocks** — it keeps the tunnel alive. The ngrok URL is printed at the bottom.

In [ ]:
!python -m liveavatar_api.examples.colab_deploy

## 8. Connect the frontend

1. Open `liveavatar_api/frontend/lite.html` locally (or any static host).
2. Paste the ngrok URL from step 7 as **Backend URL**.
3. Click **Start session**.
4. Type a viewer message → it routes through `/api/v1/lite/say` → gemma LLM → VieNeu TTS → avatar speaks.

### Debug mode (optional)

The frontend has a **Debug panel** (toggle in the UI) that lets you:
- Start a mock viewer traffic simulator (random msgs from a 200-msg pool)
- Load a mock product catalog
- Simulate random viewer counts + msg rates
- See the Director's decisions in real-time

### Engine swap (optional)

The frontend lets you swap LLM/TTS models at runtime via dropdowns.
When you select a new model, the old engine is **unloaded** (VRAM freed) before loading the new one.